IMPORT LIBRARIES

In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split , StratifiedKFold , cross_val_score
from sklearn.preprocessing import StandardScaler , LabelEncoder , MinMaxScaler , OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score , confusion_matrix
import pickle

LOAD DATASET

In [2]:
df = pd.read_csv('messy_customer_churn_10000.csv')
df

,CustomerID,Gender,SeniorCitizen,Partner,Dependents,TenureMonths,InternetService,OnlineSecurity,TechSupport,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Age,Churn
0,CUST-00001,female,NO,No,No,6.0,No,No,No internet service,monthly,NO,Mailed check,100.75,774.95,44.9,No
1,CUST-00002,Male,No,No,No,39.0,NaN,Yes,Yes,Two year,NO,Mailed check,47.84,1878.08,32.2,Yes
2,CUST-00003,MALE,No,yes,No,18.0,Fiber optic,No internet service,No internet service,month-to-month,Yes,NaN,66.44,1230.41,56.6,Yes
3,CUST-00004,M,No,NO,No,39.0,Fiber optic,No,No internet service,Month-to-month,Yes,Electronic check,75.02,3023.25,38.2,Yes
4,CUST-00005,Female,No,No,No,6.0,DSL,NO,No,Month-to-month,No,Electronic check,57.49,334.92,53.6,Churned
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10075,CUST-04370,Male,No,No,No,57.0,No,Yes,No,2-year,Yes,Credit card,80.39,4576.56,58.6,0
10076,CUST-09592,Female,yes,NaN,No,51.0,DSL,yes,No internet service,NaN,Yes,Electronic check,50.8,2405.66,55.8,No
10077,CUST-03058,Female,No,Yes,No,38.0,DSL,No,No,One year,Yes,Bank transfer,71.03,2537.32,37.7,No
10078,CUST-01368,Female,No,Yes,Yes,61.0,Fiber optic,,NO,monthly,Yes,electronic check,44.79,2433.09,46.9,Yes


EDA

In [3]:
df = df.drop('CustomerID' , axis = 1)

HANDLING AND CORRECTING INCONSISTENT DATA

In [4]:
df['Gender'] = df['Gender'].str.lower().replace({
    'm' : 'male',
    'f' : 'female'
})

In [5]:
df['SeniorCitizen'] = df['SeniorCitizen'].str.lower().replace({
    '0' : 'no',
    '1' : 'yes',
    ' ' : pd.NA
})

In [6]:
df['Partner'] = df['Partner'].str.lower().replace({
' ' : pd.NA
})

In [7]:
df['Dependents'] = df['Dependents'].str.lower().replace({
    ' ' : pd.NA
})

In [8]:
df['InternetService'] = df['InternetService'].str.lower().replace({
    ' ' : pd.NA,
    'no' : pd.NA,
    'none' : pd.NA
})

In [9]:
df['OnlineSecurity'] = df['OnlineSecurity'].str.lower().replace({
    ' ' : pd.NA,
    'no internet service' : 'no'
})

In [10]:
df['TechSupport'] = df['TechSupport'].str.lower().replace({
    ' ' : pd.NA,
    'no internet service' : 'no'
})

In [11]:
df['Contract'] = df['Contract'].str.lower().replace({
    ' ' : pd.NA,
    'two year' : 'yearly',
    'month-to-month' : 'monthly',
    'one year' : 'yearly',
    '1 year' : 'yearly',
    '2-year' : 'yearly'
})

In [12]:
df['PaperlessBilling'] = df['PaperlessBilling'].str.lower().replace({
    '1' : 'yes',
    '0' : 'no'
})

In [13]:
df['PaymentMethod'] = df['PaymentMethod'].str.lower()

In [14]:
df['MonthlyCharges'] = df['MonthlyCharges'].str.replace({
    '$' : ''
})

In [15]:
df['MonthlyCharges'] = df['MonthlyCharges'].astype(float)

In [16]:
df['Age'] = df['Age'].round().astype('Int64')

In [17]:
df['Age'] = df['Age'].replace({
    '-' : ''
})

In [18]:
df.loc[df['Age'] < 0, 'Age'] = pd.NA

In [19]:
df['Churn'] = df['Churn'].str.lower().replace({
    'churned' : 'yes',
    '1' : 'yes',
    '0' : 'no',
    ' ' : pd.NA,
    'stayed' : 'no',
    'nan' : pd.NA
})

In [20]:
df['TenureMonths'] = df['TenureMonths'].round().astype('Int64')

In [21]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10080 entries, 0 to 10079
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Gender            9145 non-null   str    
 1   SeniorCitizen     9643 non-null   str    
 2   Partner           9400 non-null   str    
 3   Dependents        9544 non-null   str    
 4   TenureMonths      9920 non-null   Int64  
 5   InternetService   7476 non-null   str    
 6   OnlineSecurity    9492 non-null   str    
 7   TechSupport       9397 non-null   str    
 8   Contract          9391 non-null   str    
 9   PaperlessBilling  9849 non-null   str    
 10  PaymentMethod     9614 non-null   str    
 11  MonthlyCharges    9937 non-null   float64
 12  TotalCharges      9666 non-null   str    
 13  Age               9847 non-null   Int64  
 14  Churn             9879 non-null   str    
dtypes: Int64(2), float64(1), str(12)
memory usage: 1.2 MB


In [22]:
df['TotalCharges'] = df['TotalCharges'].str.replace({
    ',' : ''
})

In [23]:
df['TotalCharges'] = df['TotalCharges'].astype(float)

In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10080 entries, 0 to 10079
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Gender            9145 non-null   str    
 1   SeniorCitizen     9643 non-null   str    
 2   Partner           9400 non-null   str    
 3   Dependents        9544 non-null   str    
 4   TenureMonths      9920 non-null   Int64  
 5   InternetService   7476 non-null   str    
 6   OnlineSecurity    9492 non-null   str    
 7   TechSupport       9397 non-null   str    
 8   Contract          9391 non-null   str    
 9   PaperlessBilling  9849 non-null   str    
 10  PaymentMethod     9614 non-null   str    
 11  MonthlyCharges    9937 non-null   float64
 12  TotalCharges      9666 non-null   float64
 13  Age               9847 non-null   Int64  
 14  Churn             9879 non-null   str    
dtypes: Int64(2), float64(2), str(11)
memory usage: 1.2 MB


MAKING X , Y COLUMNS

In [25]:
x = df.drop('Churn' , axis = 1)
y = df['Churn']

PREPROCESSING DATA

In [26]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [27]:
cat_features = [   'Gender' , 'SeniorCitizen' , 'Partner' , 'Dependents' , 'InternetService' , 'OnlineSecurity' , 'TechSupport' , 'Contract' , 'PaperlessBilling' , 'PaymentMethod'
    ]

In [28]:
num_features = ['TenureMonths' , 'MonthlyCharges' , 'TotalCharges' , 'Age']

In [29]:
cat_pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy='most_frequent')),
    ("encoder" , OneHotEncoder(handle_unknown='ignore'))
])

In [30]:
preprocessor = ColumnTransformer(transformers=[
    ("num_imputer" , SimpleImputer(strategy='mean'),
     ['TenureMonths' , 'MonthlyCharges' , 'TotalCharges' , 'Age']
     ),
    ("cat" , cat_pipeline , cat_features)
])

In [31]:
pipeline = Pipeline([
    ('preprocessor' , preprocessor),
    ('model' , SVC())
])

In [32]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.3 , random_state = 42 )

HANDLING MISSING VALUES IN Y COLUMN

In [33]:
mask = y_train.notna()

x_train = x_train.loc[mask]
y_train = y_train.loc[mask]

In [34]:
print("Missing values in y_train:", y_train.isna().sum())
print("Rows in x_train:", len(x_train))
print("Rows in y_train:", len(y_train))

Missing values in y_train: 0
Rows in x_train: 6924
Rows in y_train: 6924


In [35]:
cv = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state = 42
)

In [36]:
scores = cross_val_score(
    pipeline,
    x_train,
    y_train,
    cv = cv,
    scoring = 'accuracy'
)

In [37]:
pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](2,)","['no','yes']"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](14,)","['Gender','SeniorCitizen','Partner',...,'MonthlyCharges','TotalCharges', 'Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num_imputer', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remaind

SAVING THE MODEL

In [38]:
import pickle
with open("churn_model.pkl", "wb") as file:
    pickle.dump(pipeline, file)